# 01 · Environment setup (Phase 1)

**VisionTune AI**: LoRA / QLoRA fine-tuning of Qwen2.5-VL-3B-Instruct for scientific VQA.

This notebook completes the Phase 1 checklist from `docs/blueprint.md` §17:

- [ ] Verify the GPU accelerator and record GPU model / VRAM
- [ ] Record Python, PyTorch, CUDA and package versions
- [ ] Install compatible Transformers, PEFT, bitsandbytes
- [ ] Load Qwen2.5-VL-3B-Instruct and its processor
- [ ] Run inference on real ScienceQA images, and check fp16 numerics
- [ ] Check that every LoRA ablation preset resolves to the intended modules
- [ ] (Optional) Check that the 4-bit QLoRA load works
- [ ] Save environment details and test output to `/kaggle/working/phase1/`

**Kaggle settings before running:** Accelerator → *GPU T4 x2* (or any GPU); Internet → **On**.
Only one GPU is used.

In [ ]:
!nvidia-smi

## 1. Get the code and install packages

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/akhilnathreddy/VisionTune.git"
BRANCH = None  # e.g. "main"; None = the repository's default branch
REPO_DIR = "/kaggle/working/VisionTune"

if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR]
    if BRANCH:
        cmd[2:2] = ["--branch", BRANCH]
    subprocess.run(cmd, check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle images ship torch + CUDA. Upgrade only the libraries this project needs.
!pip install -q -U "transformers>=4.57" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.45" qwen-vl-utils
!pip install -q -e .

> If the next cell reports an old `transformers` version, use **Run → Restart & clear cell outputs**,
> then run from here. (The upgraded library only loads in a fresh kernel.)

In [ ]:
import json, logging
from pathlib import Path

import torch
import transformers
from visiontune.env import environment_report

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
OUT = Path("/kaggle/working/phase1")
OUT.mkdir(parents=True, exist_ok=True)

env = environment_report()
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps({k: env[k] for k in ("python", "cuda_runtime", "gpus")}, indent=2))
print({k: v for k, v in env["packages"].items() if v})

assert torch.cuda.is_available(), "No GPU: enable an accelerator in the notebook settings"
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 57), \
    "transformers is too old: restart the kernel after the pip install"
GPU = env["gpus"][0]
print(f"Using {GPU['name']} with {GPU['total_memory_gb']} GB; bf16 supported: {GPU['bf16_supported']}")

## 2. Load the processor and the base model

`model.dtype: auto` resolves to **bf16** on Ampere+ GPUs and **fp16** on T4/P100, which have no bf16 support.
The processor caps every image at 512 visual tokens (`processor.max_pixels`). `load_processor` fails
loudly if the installed transformers version ignores that cap.

In [ ]:
import time
from visiontune.config import load_config
from visiontune.models.loading import (check_visual_token_cap, count_parameters, linear_layer_summary,
                                       load_model, load_processor, resolve_dtype)

cfg = load_config("configs/base.yaml")
print("dtype ->", resolve_dtype(cfg["model"]["dtype"]))

t0 = time.perf_counter()
processor = load_processor(cfg, padding_side="left")
model = load_model(cfg, quantized=False).eval()
load_seconds = time.perf_counter() - t0

print(f"Loaded in {load_seconds:.0f}s")
print("visual tokens for a 2000x1500 image:", check_visual_token_cap(processor, cfg["processor"]["max_pixels"]))
print("parameters:", count_parameters(model))
print("linear layers:", linear_layer_summary(model))
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 3. Architecture inspection

The blueprint (§3) asks us to confirm the real module names before choosing LoRA targets. Qwen2.5-VL reuses
`gate_proj / up_proj / down_proj` inside the **vision tower** too, so `visiontune.models.targets` resolves
each ablation preset to explicit module paths. This cell shows what each preset would train.

In [ ]:
from collections import Counter
from visiontune.models.targets import PRESETS, describe_targets, resolve_target_modules

top = Counter(name.split(".")[1] if name.startswith("model.") else name.split(".")[0]
              for name, _ in model.named_parameters())
print("parameter groups:", dict(top))
for preset in PRESETS:
    names = resolve_target_modules(model, preset)
    print(f"{preset:34s} {len(names):4d} modules  {describe_targets(names)}")
print("example A1 target:", resolve_target_modules(model, "lm_attn")[0])

## 4. Inference on real ScienceQA images

This step downloads only the ScienceQA **validation** split. It runs the exact prompt used later for the
baseline (M0) and checks that the logits are finite. On T4 (fp16), non-finite values mean you should set
`model.vision_fp32: true` in `configs/base.yaml`.

In [ ]:
from datasets import load_dataset
from visiontune.data.records import scienceqa_record
from visiontune.inference.predict import Predictor, score
from visiontune.prompts import build_messages

val = load_dataset(cfg["data"]["scienceqa_id"], split="validation", revision=cfg["data"]["scienceqa_revision"])
samples = []
for i in range(len(val)):
    row = val[i]
    if row["image"] is not None:
        samples.append((scienceqa_record(row, "validation", i, row["image"]), row["image"].convert("RGB")))
    if len(samples) == 8:
        break
print("validation rows:", len(val), "| sample ids:", [r["id"] for r, _ in samples])
samples[0][1]

In [ ]:
# Numerics check: one forward pass, logits must be finite.
rec, img = samples[0]
text = processor.apply_chat_template(build_messages(rec, cfg["prompt"]["system"]), tokenize=False,
                                     add_generation_prompt=True)
enc = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
with torch.inference_mode():
    logits = model(**enc).logits
finite = bool(torch.isfinite(logits).all())
print("prompt tokens:", enc["input_ids"].shape[1], "| logits finite:", finite)
assert finite, "Non-finite logits: set model.vision_fp32=true (or use a bf16-capable GPU) and reload"
print("--- prompt ---\n" + text)

In [ ]:
predictor = Predictor(model, processor, cfg)
torch.cuda.reset_peak_memory_stats()
results = []
for rec, img in samples:                       # batch size 1 = single-request latency
    t0 = time.perf_counter()
    output = predictor.generate([rec], [img])[0]
    torch.cuda.synchronize()
    latency = time.perf_counter() - t0
    results.append({"id": rec["id"], "question": rec["question"], "choices": rec["choices"],
                    "answer_index": rec["answer_index"], "output": output, "latency_s": round(latency, 3),
                    **score(rec, output)})
    print(f"[{rec['id']}] {rec['question']}\n  gold: {'ABCDE'[rec['answer_index']]}  "
          f"pred: {results[-1]['pred_index']} ({results[-1]['extraction']})  {latency:.2f}s\n  {output[:200]!r}\n")

peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"{sum(r['correct'] for r in results)}/{len(results)} correct on this tiny smoke sample (not a benchmark)")
print(f"peak inference VRAM: {peak:.2f} GB")

In [ ]:
(OUT / "inference_smoke.json").write_text(json.dumps({
    "model": cfg["model"], "processor": cfg["processor"], "load_seconds": round(load_seconds, 1),
    "logits_finite": finite, "peak_inference_vram_gb": round(peak, 2), "results": results,
}, indent=2))
print(sorted(p.name for p in OUT.iterdir()))

## 5. (Optional) QLoRA compatibility check

This step loads the base in 4-bit NF4 with the vision tower and `lm_head` kept in 16-bit (`configs/qlora.yaml`),
then confirms which layers were quantized, attaches LoRA, and runs one forward and backward pass.
It frees the bf16/fp16 model first.

In [ ]:
import gc
from visiontune.models.loading import attach_lora
from visiontune.training.collator import SFTCollator

del model, predictor
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

qcfg = load_config("configs/qlora.yaml")
qmodel = load_model(qcfg)
print("linear layers:", linear_layer_summary(qmodel))   # expect Linear4bit only under 'language'
print(f"VRAM after 4-bit load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

qmodel = attach_lora(qmodel, qcfg)
qmodel.print_trainable_parameters()
train_processor = load_processor(qcfg, padding_side="right")
batch = SFTCollator(train_processor, qcfg["prompt"]["system"])([{"record": rec, "image": img}])
batch = {k: v.to(qmodel.device) for k, v in batch.items()}
with torch.autocast("cuda", dtype=resolve_dtype(qcfg["model"]["dtype"])):
    loss = qmodel(**batch).loss
loss.backward()
grads = [p.grad for n, p in qmodel.named_parameters() if p.requires_grad and p.grad is not None]
print(f"loss {loss.item():.4f} | LoRA tensors with gradients: {len(grads)} | "
      f"peak VRAM {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
assert torch.isfinite(loss) and grads, "QLoRA forward/backward failed"
(OUT / "qlora_check.json").write_text(json.dumps({
    "linear_layers": linear_layer_summary(qmodel), "loss": loss.item(), "lora_tensors_with_grad": len(grads),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2)}, indent=2))

## Done: Phase 1 checklist

Download `/kaggle/working/phase1/` (or **Save Version** so it is kept as notebook output), and commit
`environment.json` + `inference_smoke.json` to `results/reports/phase1/` in the repo.

**Next:** `02_dataset_exploration.ipynb`, which runs `python -m visiontune.data.prepare --config configs/base.yaml`
and inspects the cleaned splits.